# Treino e avaliação — LSTM D+1 (PETR4.SA)

**Treino oficial (reproduzível):** `python -m src.model.train`  
**Serviço de previsão:** API FastAPI (`uvicorn` / Docker), não este notebook.

Este arquivo só **explica** o modelo e **mostra** as métricas já gravadas em `models/metrics.json`, comparando o LSTM com a baseline naive (amanhã = último fechamento da janela).

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.settings import MODELS_DIR, METRICS_FILENAME, MODEL_FILENAME, SCALER_FILENAME

metrics_path = MODELS_DIR / METRICS_FILENAME
model_path = MODELS_DIR / MODEL_FILENAME
scaler_path = MODELS_DIR / SCALER_FILENAME
print("métricas:", metrics_path, "existe:" , metrics_path.exists())
print("modelo:", model_path.exists(), "scaler:", scaler_path.exists())
if not metrics_path.exists():
    raise FileNotFoundError(
        "Rode o treino oficial antes: python -m src.model.train"
    )

## Métricas no teste (preço em R$)

MAE, RMSE e MAPE depois de inverter o scaler. A naive costuma ganhar em preço absoluto — isso é esperado e faz parte da discussão da entrega.

In [ ]:
import pandas as pd

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
table = pd.DataFrame(metrics).T[["mae", "rmse", "mape"]]
table.columns = ["MAE", "RMSE", "MAPE (%)"]
table

## LSTM em uma frase

Duas camadas LSTM + Dropout + Dense(1). Entrada: 60 fechamentos normalizados. Saída: **um** fechamento (D+1). O scaler foi ajustado **só no treino**.

## Previsão D+1 versus fechamento real (teste)

Usa o modelo e o scaler persistidos e as janelas de `src.data` — sem reimplementar o pipeline.

In [ ]:
import matplotlib.pyplot as plt

from src.data import collect_prices, clean_series, chronological_split, transform_close, make_windows
from src.model.artifacts import invert_scale, load_artifacts

if not (model_path.exists() and scaler_path.exists()):
    raise FileNotFoundError(
        "Modelo ou scaler ausentes. Treino oficial: python -m src.model.train"
    )

model, scaler = load_artifacts()
cleaned = clean_series(collect_prices())
_train, _val, test = chronological_split(cleaned)
x_test, y_test, insufficient = make_windows(
    transform_close(test, scaler), report_insufficient=False
)
if insufficient:
    raise RuntimeError("conjunto de teste curto demais para janelas 60 → 1")

pred_scaled = model.predict(x_test, verbose=0).ravel()
y_true = invert_scale(scaler, y_test)
y_lstm = invert_scale(scaler, pred_scaled)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_true, label="Fechamento real", color="#333333")
ax.plot(y_lstm, label="LSTM D+1", color="#1f77b4", alpha=0.85)
ax.set_title("Teste: real vs previsão de um pregão à frente")
ax.set_xlabel("Janela no conjunto de teste")
ax.set_ylabel("Close (R$)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

Se o LSTM parecer um atraso suavizado da série, combine com a tabela da naive. A API (`POST /predict`) é o único serviço de previsão da entrega.